<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [1]:
#%pip install langchain langchain_community pypdf
#%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')


C:\Users\Andrey\AppData\Local\Temp\ipykernel_37348\3562358464.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [7]:
# File path for the document
file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [8]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load()
len(pages)

310

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [23]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [24]:
load_dotenv()

True

In [25]:
import os 
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [26]:
vector = embeddings.embed_query("What is artificial intelligence?")

print("Vector dimensions:", len(vector))
print("First 5 values:", vector[:5])

Vector dimensions: 3072
First 5 values: [-0.00890350341796875, -0.0029239654541015625, -0.0174560546875, -0.0014753341674804688, 0.02349853515625]


<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [28]:
from langchain_chroma import Chroma

In [29]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


In [30]:
print("Stored chunks:", db._collection.count())

Stored chunks: 1096


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [31]:
user_question = "What are the main applications of artificial intelligence?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [32]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
 combustion engine. While this remains 
to be seen, AI has found diverse applications from recommendation engines 
and targeted advertising to predictive policing software, predictive mainte -
nance, customer resource management and fraud detection. Capital became 
visibly interested in machine learning around 2015. All the biggest tech com -
panies in the world have since shifted to AI-intensive directions, including
Document 2:
For some years now, technical solutions utilising artificial intelligence are 
widely seen as means to tackle many fundamental problems of mankind. From 
fighting the climate crisis, tackling the problems of ageing societies, reducing 
global poverty, stopping terror, detecting copyright infringements or curing 
cancer to improving evidence-based politics, improving predictive police work, 
local transportation, self-driving cars and even waste removal.
Definitions
The first step towards a meaningful discussion about AI would be to define 
what exa

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [33]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [34]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [35]:
prompt = f"""
You are an AI education assistant specializing in
explaining artificial intelligence to beginners.

Use the provided context from the book "AI for Everyone"
to answer the user's question.

Instructions:
1. Answer using only the provided context.
2. Explain concepts in simple, clear language.
3. Provide practical examples when supported by the context.
4. Structure your answer using Markdown.
5. If the answer is not found in the context,
   clearly state that the information is unavailable.
6. Do not invent facts or information.

Context:
{formatted_context}

User Question:
{user_question}

Answer:

"""


In [36]:
print(prompt[:1500])


You are an AI education assistant specializing in
explaining artificial intelligence to beginners.

Use the provided context from the book "AI for Everyone"
to answer the user's question.

Instructions:
1. Answer using only the provided context.
2. Explain concepts in simple, clear language.
3. Provide practical examples when supported by the context.
4. Structure your answer using Markdown.
5. If the answer is not found in the context,
   clearly state that the information is unavailable.
6. Do not invent facts or information.

Context:


Content:
purpose technology comparable to the combustion engine. While this remains 
to be seen, AI has found diverse applications from recommendation engines 
and targeted advertising to predictive policing software, predictive mainte -
nance, customer resource management and fraud detection. Capital became 
visibly interested in machine learning around 2015. All the biggest tech com -
panies in the world have since shifted to AI-intensive directio

In [37]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [39]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.3 ,  # Increase creativity
    'max_tokens': 1000,  # Allow for longer responses
    'top_p': 1,        # Use nucleus sampling
    'frequency_penalty': 0,  # Reduce repetition
    'presence_penalty': 0   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [40]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [41]:
answer = completion.choices[0].message.content
print(answer)

The main applications of artificial intelligence, as mentioned in the provided context, include:

1. **Recommendation Engines**: AI is used to suggest products, services, or content to users based on their preferences and past behavior.

2. **Targeted Advertising**: AI helps in delivering personalized advertisements to users, increasing the effectiveness of marketing campaigns.

3. **Predictive Policing Software**: AI is applied to predict and prevent crimes by analyzing data and identifying patterns.

4. **Predictive Maintenance**: AI systems are used to predict equipment failures before they occur, allowing for timely maintenance and reducing downtime.

5. **Customer Resource Management**: AI assists in managing customer relationships by analyzing data and providing insights to improve customer service.

6. **Fraud Detection**: AI is employed to identify fraudulent activities by analyzing transaction patterns and detecting anomalies.

7. **Fighting Climate Crisis**: AI solutions are 

In [ ]:
from IPython.display import Markdown, display
display(Markdown(answer))

The main applications of artificial intelligence, as mentioned in the provided context, include:

1. **Recommendation Engines**: AI is used to suggest products, services, or content to users based on their preferences and past behavior.

2. **Targeted Advertising**: AI helps in delivering personalized advertisements to users, increasing the effectiveness of marketing campaigns.

3. **Predictive Policing Software**: AI is applied to predict and prevent crimes by analyzing data and identifying patterns.

4. **Predictive Maintenance**: AI systems are used to predict equipment failures before they occur, allowing for timely maintenance and reducing downtime.

5. **Customer Resource Management**: AI assists in managing customer relationships by analyzing data and providing insights to improve customer service.

6. **Fraud Detection**: AI is employed to identify fraudulent activities by analyzing transaction patterns and detecting anomalies.

7. **Fighting Climate Crisis**: AI solutions are being explored to address environmental challenges and promote sustainability.

8. **Tackling Ageing Societies**: AI is used to address issues related to ageing populations, such as healthcare and social support.

9. **Reducing Global Poverty**: AI applications aim to alleviate poverty through various means, including improving access to resources and education.

10. **Stopping Terror**: AI is utilized in security and surveillance to prevent terrorist activities.

11. **Detecting Copyright Infringements**: AI helps in identifying unauthorized use of intellectual property.

12. **Curing Cancer**: AI is being researched for its potential to improve cancer diagnosis and treatment.

13. **Improving Evidence-Based Politics**: AI can analyze data to support informed decision-making in politics.

14. **Local Transportation and Self-Driving Cars**: AI is crucial in developing autonomous vehicles and improving transportation systems.

15. **Waste Removal**: AI technologies are being explored to optimize waste management processes.

These applications demonstrate the diverse and impactful ways AI is being integrated into various sectors to solve complex problems and improve efficiency.

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [43]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [46]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold'],force_color=True))
    return text

### Exercice4: add your keywords

In [47]:
query_keywords = [] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
purpose technology comparable to the combustion engine. While this remains 
to be seen, AI has found diverse applications from recommendation engines 
and targeted advertising to predictive policing s
--------------------------------------------------------------------------------


In [48]:
text = "Artificial intelligence and machine learning are changing the world."

keywords = [
    "Artificial intelligence",
    "machine learning"
]

highlighted_text = highlight_keywords(text, keywords)

print(highlighted_text)

Artificial intelligence and machine learning are changing the world.


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:
